In [ ]:
import sys
import os
import yaml

In [ ]:
notebook_dir = os.getcwd()
src_dir = notebook_dir  # The notebook is already in src/
sys.path.insert(0, src_dir)

print(f"Added to path: {src_dir}")
print(f"Current working directory: {notebook_dir}")

In [ ]:
from app.trainer import OnPolicyTrainer, OnPolicySchedule

# Actor Critic

In [1]:
from app.rl_agents import ActorCritic
from app.models import ValueModel, StochasticDiscretePolicy
from app.schedulers import ScheduleWrapper
from app.normalizer import RunningNorm, BatchNorm
from app.env_wrapper import GymnasiumWrapper
from app.buffer import RolloutBuffer
from app.renderer import Renderer
from app.rl_callbacks import WandbCallback
from app.logging_config import configure_logging

e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Reacher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Pusher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedPendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedDoublePendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\env

In [ ]:
import numpy as np
np.__version__

In [ ]:
# Configure logger
app_logger = configure_logging()

In [2]:
# Create Env
env = GymnasiumWrapper(
    cfg='LunarLanderContinuous-v3',
    num_envs=8,
    wrappers=[],
    render_mode=None,
    seed=42,
    obs_key=None,
    goal_key=None,
    ach_goal_key=None
    )

In [3]:
env.finite_horizon

True

In [ ]:
env.observation_space.sample()

In [ ]:
# Create policy
policy = StochasticDiscretePolicy(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.001}
    },
    lr_scheduler=None,
    distribution='categorical',
    device='cuda'
)

In [ ]:
# Create value model
value = ValueModel(
    env=env,
    layer_config=[
        {
            'type': 'dense',
            'params': {'units': 64, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        },
        {
            'type': 'dense',
            'params': {'units': 32, 'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    output_config=[
        {
            'type': 'dense',
            'params': {'kernel': 'orthogonal', 'kernel_params': {'gain': 1.0}}
        }
    ],
    optimizer_params={
        'type': 'Adam',
        'params': {'lr': 0.0001}
    },
    lr_scheduler=None,
    device='cuda'
)

In [ ]:
# Create Normalizers
state_normalizer = Normalizer(
    size=4,
    device='cuda'
)

advantage_normalizer = Normalizer(
    size=1,
    device='cuda'
)

In [ ]:
# Set params
discount = 0.99
policy_trace_decay = 0.0
value_trace_decay = 0.0
entropy_coefficient = 0.0
entropy_schedule = None
gae_coefficient = 0.95
save_dir = "E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Trained_Models/Test_CartPole-v1_1/Reinforce/"
device = "cuda"

In [ ]:
# Build ActorCritic
agent = ActorCritic(
    policy=policy,
    value=value,
    discount=discount,
    policy_trace_decay=policy_trace_decay,
    value_trace_decay=value_trace_decay,
    entropy_coefficient=entropy_coefficient,
    # entropy_schedule=entropy_schedule,
    gae_coefficient=gae_coefficient,
    state_normalizer=state_normalizer,
    advantage_normalizer=advantage_normalizer,
    save_dir=save_dir,
    device=device
)


In [ ]:
# Create Buffer
buffer = RolloutBuffer(
    env=env,
    buffer_size=100,
    device=device
)

In [ ]:
# Create Schedule
schedule = OnPolicySchedule(
    unit='episode',
    num_units=1000,
    learn_unit='timestep',
    num_learn_units=800,
    seed=42
)


In [ ]:
# Create Renderer
renderer = Renderer(
    render_freq=1000,
    save_dir=save_dir,
    fps=30,
    codec='libx264'
)


In [ ]:
# Create callbacks
callbacks = [
    WandbCallback(
        project_name="CartPole-v1"
    )
]

In [ ]:
# Create trainer
trainer = OnPolicyTrainer(
    agent=agent,
    env=env,
    buffer=buffer,
    schedule=schedule,
    renderer=renderer,
    callbacks=callbacks
)

In [ ]:
trainer.train()

In [ ]:
trainer.buffer.states.shape

# Test agent.py

In [ ]:
from scripts.agent import build_trainer_from_config_path as build

In [ ]:
trainer = build("E:/Documents/Programming/Projects/Reinforcement/PhoenX_RL/src/Configs/actor_critic.yml")

In [ ]:
trainer.renderer

# Smooth Surrogate Testing

In [ ]:
import torch as T
import torch.nn.functional as F

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=True, k=15.0):
    """
    ratio: tensor of π_new / π_old, shape advantages: tensor of A_t, shape epsilon: clip param, like 0.2
    smooth: bool, if True use soft clip instead of hard min/max
    k: sharpness of the soft curve (higher = sharper, like hard clip)
    
    Returns: clipped surrogate objective (scalar or per-sample)
    """
    if not smooth:
        # Vanilla PPO: hard clip
        clipped_ratio = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        surrogate = clipped_ratio * advantages
        return surrogate.mean()  # or .sum(), whatever your loss needs
    
    else:
        # Soft clip: gentle curve past bounds
        # We focus on upper bound here (r > 1+ε); lower is symmetric
        upper = 1 + epsilon
        print(f'upper:{upper}')
        # Softplus version: smooth ramp-down after upper
        # sigmoid(k*(r - upper)) starts at 1 when r=upper, drops to 0 as r grows
        decay = T.sigmoid(k * (ratio - upper))  # 0 to 1, smooth
        print(f'decay:{decay}')
        
        # Effective ratio: caps at upper, but adds a decaying tail
        effective_ratio = upper + (ratio - upper) * decay
        print(f'ratio-upper:{ratio-upper}')
        print(f'added ratio:{(ratio-upper)*decay}')
        print(f'effective ratio:{effective_ratio}')
        # Clamp lower too, for symmetry (optional but nice)
        effective_ratio = T.clamp(effective_ratio, 1 - epsilon, upper + 0.3)  # small overshoot
        print(f'effective lower clamp ratio:{effective_ratio}')
        surrogate = effective_ratio * advantages
        return surrogate.mean()

In [ ]:
advantages = T.tensor([1, 10, 20])
ratio = T.tensor([1.8])
epsilon = T.tensor([0.2])
k = 10

surrogates = ppo_surrogate(ratio, advantages, epsilon, k=k)
print(f'surrogates:{surrogates}')

In [ ]:
import torch

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=False, k=12.0):
    """
    ratio: tensor of π_new / π_old
    advantages: tensor of A_t
    smooth=True → soft clipping (no zero gradients past bounds)
    k: higher = sharper decay (closer to hard clip)
    """
    if not smooth:
        # Original hard PPO clip
        clipped = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        return (clipped * advantages).mean()
    
    # === Soft version (only decays outside the clip window) ===
    lower = 1.0 - epsilon
    upper = 1.0 + epsilon
    effective = ratio.clone()
    
    # Upper side (r > 1+ε): overshoot a little, then decay back to 1+ε
    excess = T.clamp(ratio - upper, min=0.0)
    print(f'excess:{excess}')
    decay_upper = T.exp(-k * excess)
    print(f'decay_upper:{decay_upper}')
    effective = T.where(ratio > upper,
                            upper + excess * decay_upper,
                            effective)
    print(f'effective{effective}')
    
    # Lower side (r < 1-ε): undershoot a little, then decay back to 1-ε
    deficit = T.clamp(lower - ratio, min=0.0)
    print(f'deficit:{deficit}')
    decay_lower = T.exp(-k * deficit)
    print(f'decay_lower:{decay_lower}')
    effective = T.where(ratio < lower,
                            lower - deficit * decay_lower,
                            effective)
    print(f'effective{effective}')
    
    surrogate = effective * advantages
    return surrogate.mean()

In [ ]:
advantages = T.tensor([1])
ratio = T.tensor([0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6])
epsilon = T.tensor([0.2])
k = 15

surrogates = ppo_surrogate(ratio, advantages, epsilon, True, k=k)

In [ ]:
import torch as T

def ppo_surrogate(ratio, advantages, epsilon=0.2, smooth=False, k=10.0):
    """
    ratio: torch.Tensor of π_new / π_old
    advantages: torch.Tensor of A_t
    smooth=True → soft clipping (no zero gradients past bounds)
    k: controls how fast the overshoot tapers (higher k = closer to hard clip)
       Try 5–20. Default 10 is nice.
    """
    if not smooth:
        # Vanilla hard PPO clip
        clipped = T.clamp(ratio, 1 - epsilon, 1 + epsilon)
        return (clipped * advantages).mean()
    
    # === Monotonic soft clip (never decreases after crossing bound) ===
    lower = 1.0 - epsilon
    upper = 1.0 + epsilon
    effective = ratio.clone()
    
    # Upper side (r > 1+ε): gentle continued growth that slows down
    excess = T.clamp(ratio - upper, min=0.0)
    print(f'excess:{excess}')
    soft_excess = excess / (1.0 + k * excess)          # ← this is the key line
    print(f'soft_excess:{soft_excess}')
    effective = T.where(ratio > upper,
                            upper + soft_excess,
                            effective)
    print(f'effective:{effective}')
    
    # Lower side (r < 1-ε): symmetric
    deficit = T.clamp(lower - ratio, min=0.0)
    print(f'deficit:{deficit}')
    soft_deficit = deficit / (1.0 + k * deficit)
    print(f'soft_deficit:{soft_deficit}')
    effective = T.where(ratio < lower,
                            lower - soft_deficit,
                            effective)
    print(f'effective:{effective}')
    
    surrogate = effective * advantages
    return surrogate.mean()

In [ ]:
advantages = T.tensor([1])
ratio = T.tensor([0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8])
epsilon = T.tensor([0.2])
k = 15

surrogates = ppo_surrogate(ratio, advantages, epsilon, True, k=k)